In [1]:
%pip install pandas sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

usuario = "root"
contrasena = quote_plus(input("Escribe tu contraseña de MySQL: "))
host = "localhost"
base_datos = "sakila"

engine = create_engine(
    f"mysql+pymysql://{usuario}:{contrasena}@{host}:3306/{base_datos}"
)

with engine.connect() as conexion:
    print("¡Conexión correcta con Sakila!")

Escribe tu contraseña de MySQL:  ABUelo


¡Conexión correcta con Sakila!


In [3]:
def rentals_month(engine, month, year):
    query = f"""
    SELECT *
    FROM rental
    WHERE MONTH(rental_date) = {int(month)}
      AND YEAR(rental_date) = {int(year)}
    """
    return pd.read_sql(query, engine)

In [4]:
rentals_may = rentals_month(engine, 5, 2005)

rentals_may.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [5]:
def rental_count_month(rentals, month, year):
    column_name = f"rentals_{month:02d}_{year}"

    rental_counts = (
        rentals.groupby("customer_id")
        .size()
        .reset_index(name=column_name)
    )

    return rental_counts

In [6]:
may_counts = rental_count_month(rentals_may, 5, 2005)

may_counts.head()

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


In [7]:
rentals_june = rentals_month(engine, 6, 2005)

rentals_june.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1158,2005-06-14 22:53:33,1632,416,2005-06-18 21:37:33,2,2006-02-15 21:30:53
1,1159,2005-06-14 22:55:13,4395,516,2005-06-17 02:11:13,1,2006-02-15 21:30:53
2,1160,2005-06-14 23:00:34,2795,239,2005-06-18 01:58:34,2,2006-02-15 21:30:53
3,1161,2005-06-14 23:07:08,1690,285,2005-06-21 17:12:08,1,2006-02-15 21:30:53
4,1162,2005-06-14 23:09:38,987,310,2005-06-23 22:00:38,1,2006-02-15 21:30:53


In [8]:
june_counts = rental_count_month(rentals_june, 6, 2005)

june_counts.head()

,customer_id,rentals_06_2005
0,1,7
1,2,1
2,3,4
3,4,6
4,5,5


In [10]:
def compare_rentals(df1, df2):
    comparison = pd.merge(
        df1,
        df2,
        on="customer_id",
        how="outer"
    ).fillna(0)

    comparison["rentals_05_2005"] = (
        comparison["rentals_05_2005"].astype(int)
    )
    comparison["rentals_06_2005"] = (
        comparison["rentals_06_2005"].astype(int)
    )

    comparison["difference"] = (
        comparison["rentals_06_2005"]
        - comparison["rentals_05_2005"]
    )

    return comparison

In [11]:
comparison = compare_rentals(may_counts, june_counts)

comparison.head(10)

,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2,7,5
1,2,1,1,0
2,3,2,4,2
3,4,0,6,6
4,5,3,5,2
5,6,3,4,1
6,7,5,5,0
7,8,1,3,2
8,9,3,2,-1
9,10,1,5,4
